# Week 3, Day 1: A wrestling poster and a ring announcer

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1DjcrYDZldAXKJ08x1uYIVCtItoLPk1Wr) linked from [week3/day1.ipynb](../../../week3/day1.ipynb), *Introducing Colab*. **About 40 minutes.**

**What you practice here**

- Logging in to Hugging Face with a token kept as a secret, as every Colab of the week does.
- Loading a text-to-image model with diffusers in half precision, moving it to the GPU, and making an image with a number of steps and a guidance scale, including SDXL's handoff from its base model to its refiner.
- Calling a text-to-speech pipeline with a speaker's voice, and reading the audio and its sampling rate from what it returns.
- In Try it for real: working on a Colab T4, checking the GPU, and restarting the session to free its memory.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week3/day1", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson shows what a free Colab GPU can do with open models: it logs in to Hugging Face, makes images with
diffusers, and speaks a line with a text-to-speech pipeline. Build the same kit for a wrestling show: a poster for a
1980s stadium event, and the ring announcer's call.

The libraries (`huggingface_hub`, `diffusers`, `transformers` and `torch`) are not in the checks' Python. So each
function takes the library object it uses as its first parameter, such as the `login` function, a diffusers class or a
loaded pipeline, and calls it exactly as the lesson does. The checks hand your functions stand-ins that record each
call and answer with canned results. In Colab, you pass the real ones.

## What each call should give

```python
hf_login(login, userdata.get)
# reads the secret HF_TOKEN, checks that it looks like a Hugging Face token, and logs in with it

pipe = load_text_to_image(AutoPipelineForText2Image, torch.float16)
# loads stabilityai/sdxl-turbo in float16, from its fp16 weights, and moves it to "cuda"

make_image(pipe, "A 1980s wrestling poster for a sold-out stadium show, vibrant pop-art style")
# calls the pipeline once with the prompt, 4 steps and no guidance, and returns the first image

refine(base, refiner, "A 1980s wrestling poster for a sold-out stadium show, vibrant pop-art style")
# the base runs the first 80% of 40 steps and hands its latents to the refiner, which finishes the image

audio, rate = announce(synthesizer, "Ladies and gentlemen, the following contest is scheduled for one fall!", voice)
# speaks the line in that voice: audio holds the sound, and rate is its sampling rate, such as 16000
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `hf_login(login, get_secret)`

Logs in to Hugging Face with the token kept in the secret `HF_TOKEN`. `login` is Hugging Face's login function, and
`get_secret` is a function that returns a secret by its name: Colab's `userdata.get`, or `os.environ.get` in your
codespace. Read the secret `HF_TOKEN` with `get_secret`. If it is missing or empty, or does not start with `hf_`,
raise `ValueError` with a message that says what is wrong, and do not log in. Otherwise log in once with the token,
with `add_to_git_credential=True` as the lesson does, and return nothing.

<details><summary>Hint</summary>

```python
token = get_secret("HF_TOKEN")
# ... raise ValueError when it is missing, empty, or does not start with "hf_"
login(token, add_to_git_credential=True)
```

</details>

In [ ]:
def hf_login(login, get_secret):
    # Write hf_login here, as described above.
    raise NotImplementedError("hf_login is not written yet")

In [ ]:
check("hf_login")

### 2 · `load_text_to_image(pipeline_class, dtype, model="stabilityai/sdxl-turbo", device="cuda")`

Loads a text-to-image model in half precision and moves it to the GPU. `pipeline_class` is a diffusers class such as
`AutoPipelineForText2Image`, and `dtype` is a torch dtype such as `torch.float16`: both are passed in because the
checks have neither diffusers nor torch. Load `model` once with the class's loading method, with `dtype` as the
precision and the `"fp16"` variant of the weights, as the lesson does. Then move the pipeline to `device` and return
it; moving a diffusers pipeline returns the pipeline itself.

<details><summary>Hint</summary>

```python
pipe = pipeline_class.from_pretrained(model, torch_dtype=dtype, variant="fp16")
return pipe.to(device)
```

</details>

In [ ]:
def load_text_to_image(pipeline_class, dtype, model="stabilityai/sdxl-turbo", device="cuda"):
    # Write load_text_to_image here, as described above.
    raise NotImplementedError("load_text_to_image is not written yet")

In [ ]:
check("load_text_to_image")

### 3 · `make_image(pipe, prompt, steps=4, guidance_scale=0.0)`

Makes one image from a prompt and returns it. Call the pipeline once with the prompt, `steps` as the number of
inference steps, and `guidance_scale` as the guidance scale, every setting by name. Return the first image of the
result. The defaults are the lesson's settings for sdxl-turbo: 4 steps and no guidance.

<details><summary>Hint</summary>

```python
pipe(prompt=prompt, num_inference_steps=steps, guidance_scale=guidance_scale).images[0]
```

</details>

In [ ]:
def make_image(pipe, prompt, steps=4, guidance_scale=0.0):
    # Write make_image here, as described above.
    raise NotImplementedError("make_image is not written yet")

In [ ]:
check("make_image")

### 4 · `refine(base, refiner, prompt, n_steps=40, high_noise_frac=0.8)`

Makes one image with SDXL's two experts: the base model, then the refiner. Call the base once with the prompt and
`n_steps` inference steps, telling it to stop denoising at `high_noise_frac` and to return latents rather than
pictures. Keep the images of its result as they are: they are latents, the half-made picture, so do not take the
first one. Then call the refiner once with the same prompt and number of steps, telling it to start denoising at
`high_noise_frac`, with those latents as its image, and return the refiner's first image. With the defaults, the base
runs the first 80% of the 40 steps and the refiner runs the rest.

<details><summary>Hint</summary>

```python
latents = base(prompt=prompt, num_inference_steps=n_steps, denoising_end=high_noise_frac,
               output_type="latent").images
return refiner(prompt=prompt, num_inference_steps=n_steps, denoising_start=high_noise_frac,
               image=latents).images[0]
```

</details>

In [ ]:
def refine(base, refiner, prompt, n_steps=40, high_noise_frac=0.8):
    # Write refine here, as described above.
    raise NotImplementedError("refine is not written yet")

In [ ]:
check("refine")

### 5 · `announce(synthesizer, text, speaker_embedding)`

Speaks a line with a text-to-speech pipeline and returns the sound with its sampling rate. `synthesizer` is the
speecht5 text-to-speech pipeline the lesson makes, and `speaker_embedding` is one speaker's voice. Call the pipeline
once on `text`, passing the voice as the forward parameter `speaker_embeddings`, as the lesson does. The pipeline
returns a dict: return the tuple of its audio and its sampling rate, ready for `Audio(audio, rate=rate)`.

<details><summary>Hint</summary>

```python
speech = synthesizer(text, forward_params={"speaker_embeddings": speaker_embedding})
return speech["audio"], speech["sampling_rate"]
```

</details>

In [ ]:
def announce(synthesizer, text, speaker_embedding):
    # Write announce here, as described above.
    raise NotImplementedError("announce is not written yet")

In [ ]:
check("announce")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 3, Day 1 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week3/day1.ipynb), choose a T4 runtime, and run the cells from the top. The first cell fetches the checker from your fork, so the checks run in Colab too. Write the steps in the 4 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

Each model downloads a few GB the first time, and Colab's free T4 holds only one of them at a time: restart the
session between them, as the lesson does. Add your Hugging Face token to Colab's secrets as `HF_TOKEN` first, with
notebook access on.

1. In the first Try cell, keep the `!pip install` line, then check the GPU as the lesson does, and log in with
   `hf_login(login, userdata.get)`.
2. Load sdxl-turbo with `load_text_to_image(AutoPipelineForText2Image, torch.float16)`, and display a poster that
   `make_image` makes for a 1980s stadium wrestling show.
3. Free the GPU: Runtime > Restart session, then run the cells from the top again, but skip step 2. Make the
   speecht5 pipeline, take speaker 7306's x-vector from the `matthijs/cmu-arctic-xvectors` dataset, and play the ring
   announcer's call from `announce` with `Audio(audio, rate=rate)`.
4. Optional: restart the session again, and run the cells from the top but skip steps 2 and 3, since sdxl-turbo,
   speecht5, and SDXL's base with its refiner do not all fit on a T4 together. Load SDXL's base with
   `load_text_to_image(DiffusionPipeline, torch.float16, model="stabilityai/stable-diffusion-xl-base-1.0")` and its
   refiner as the lesson does, and compare `refine`'s poster with sdxl-turbo's.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2 datasets==3.6.0
import subprocess
from google.colab import userdata
from huggingface_hub import login

# the lesson's gpu_info = !nvidia-smi works in Colab too; subprocess does the same in plain Python
smi = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
gpu_info = smi.stdout + smi.stderr
if smi.returncode or "failed" in gpu_info:
    print("Not connected to a GPU")
elif "Tesla T4" in gpu_info:
    print("Success - connected to a T4")
else:
    print("Connected to a GPU, but not to a T4")
hf_login(login, userdata.get)
```

```python
import torch
from diffusers import AutoPipelineForText2Image
from IPython.display import display

POSTER = "A 1980s wrestling poster for a sold-out stadium show, a ring under bright lights, vibrant pop-art style"
pipe = load_text_to_image(AutoPipelineForText2Image, torch.float16)
display(make_image(pipe, POSTER))
```

```python
import torch
from datasets import load_dataset
from IPython.display import Audio
from transformers import pipeline

synthesizer = pipeline("text-to-speech", "microsoft/speecht5_tts", device="cuda")
xvectors = load_dataset("matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
voice = torch.tensor(xvectors[7306]["xvector"]).unsqueeze(0)
audio, rate = announce(synthesizer, "Ladies and gentlemen, the following contest is scheduled for one fall!", voice)
Audio(audio, rate=rate)
```

```python
import torch
from diffusers import DiffusionPipeline
from IPython.display import display

POSTER = "A 1980s wrestling poster for a sold-out stadium show, a ring under bright lights, vibrant pop-art style"
base = load_text_to_image(DiffusionPipeline, torch.float16, model="stabilityai/stable-diffusion-xl-base-1.0")
refiner = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-refiner-1.0",
                                            text_encoder_2=base.text_encoder_2, vae=base.vae,
                                            torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
refiner = refiner.to("cuda")
display(refine(base, refiner, POSTER))
```

</details>

In [ ]:
!pip install -q --upgrade transformers==4.56.2 diffusers==0.32.2 datasets==3.6.0
# Step 1: check the GPU, and log in to Hugging Face with hf_login

In [ ]:
# Step 2: the poster, with sdxl-turbo

In [ ]:
# Step 3, after Runtime > Restart session and running the cells from the top again (but not step 2): the ring announcer

In [ ]:
# Step 4 (optional), after another restart and running the cells from the top again (but not steps 2 and 3): SDXL's base and refiner